<a href="https://colab.research.google.com/github/MNMARQUES/MachineLearning/blob/main/Projeto3/EML_PROJETO3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Instalação, se necessário
!pip install -q scikit-learn pandas scipy

import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml

# Carregar o dataset OpenML 1480
dataset = fetch_openml(data_id=1480, as_frame=True)

X = dataset.data.copy()
y = dataset.target.copy()

print("Nome do dataset:", dataset.details.get("name"))
print("Número de amostras:", X.shape[0])
print("Número de features:", X.shape[1])
print("\nDimensões de X:", X.shape)
print("Dimensões de y:", y.shape)

display(X.head())
print("\nClasses da variável-alvo:")
print(y.value_counts(dropna=False))


Nome do dataset: ilpd
Número de amostras: 583
Número de features: 10

Dimensões de X: (583, 10)
Dimensões de y: (583,)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40



Classes da variável-alvo:
Class
1    416
2    167
Name: count, dtype: int64


In [2]:

# Tipos das features
print("Tipos de dados:")
print(X.dtypes)

# Quantidade de valores ausentes
print("\nValores ausentes por feature:")
print(X.isnull().sum())

print("\nTotal de valores ausentes:", X.isnull().sum().sum())

# Identificar variáveis categóricas
categoricas = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

print("\nFeatures categóricas:", categoricas)

# Mapear V2, conforme solicitado
if "V2" in X.columns:
    X["V2"] = X["V2"].replace({
        "Female": 0,
        "Male": 1
    })

    print("\nValores de V2 após o mapeamento:")
    print(X["V2"].value_counts(dropna=False))
else:
    print("\nA coluna V2 não foi encontrada. Confira os nomes das colunas.")


Tipos de dados:
V1        int64
V2     category
V3      float64
V4      float64
V5        int64
V6        int64
V7        int64
V8      float64
V9      float64
V10     float64
dtype: object

Valores ausentes por feature:
V1     0
V2     0
V3     0
V4     0
V5     0
V6     0
V7     0
V8     0
V9     0
V10    0
dtype: int64

Total de valores ausentes: 0

Features categóricas: ['V2']

Valores de V2 após o mapeamento:
V2
1    441
0    142
Name: count, dtype: int64


/tmp/ipykernel_533/4171318118.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["V2"] = X["V2"].replace({
/tmp/ipykernel_533/4171318118.py:20: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X["V2"] = X["V2"].replace({


In [3]:

from scipy.stats import uniform, randint

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.metrics import f1_score, make_scorer

# Codificar a variável-alvo, caso seja categórica
if y.isnull().any():
    raise ValueError("A variável-alvo possui valores ausentes. Investigue antes de treinar.")

le = LabelEncoder()
y_encoded = le.fit_transform(y)

if len(np.unique(y_encoded)) != 2:
    raise ValueError(
        f"O desafio usa F1 binário, mas foram encontradas "
        f"{len(np.unique(y_encoded))} classes. Confira a variável-alvo."
    )

# F1 binário, conforme solicitado
f1 = make_scorer(f1_score)

# Validação cruzada estratificada com 10 folds
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=5762
)

print("Validação cruzada configurada.")
print("Número de folds:", skf.get_n_splits())
print("Classes originais:", list(le.classes_))


Validação cruzada configurada.
Número de folds: 10
Classes originais: ['1', '2']


In [4]:

# Identificar colunas depois do mapeamento de V2
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

numeric_svc = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_svc = ColumnTransformer([
    ("num", numeric_svc, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

preprocessor_rf = ColumnTransformer([
    ("num", numeric_rf, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


In [5]:

# Pipeline do SVC
svc_pipeline = Pipeline([
    ("preprocessor", preprocessor_svc),
    ("model", SVC())
])

# Espaço de hiperparâmetros
param_dist_svc = {
    "model__kernel": ["sigmoid", "poly", "rbf"],
    "model__C": uniform(loc=1, scale=9)
}

# Busca aleatória
random_search_svc = RandomizedSearchCV(
    estimator=svc_pipeline,
    param_distributions=param_dist_svc,
    n_iter=5,
    scoring=f1,
    cv=skf,
    random_state=5762,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

# Treinamento e busca
random_search_svc.fit(X, y_encoded)

# Resultados
print("SVC - Melhor F1:", random_search_svc.best_score_)
print("SVC - Melhores parâmetros:", random_search_svc.best_params_)
print("SVC - Melhor estimador:")
print(random_search_svc.best_estimator_)


SVC - Melhor F1: 0.2030825744304005
SVC - Melhores parâmetros: {'model__C': np.float64(5.539145131649216), 'model__kernel': 'rbf'}
SVC - Melhor estimador:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['V1', 'V3', 'V4', 'V5', 'V6',
                                                   'V7', 'V8', 'V9', 'V10']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
       

In [6]:

# Pipeline do Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("model", RandomForestClassifier(random_state=5762))
])

# Espaço de hiperparâmetros
param_dist_rf = {
    "model__n_estimators": randint(10, 1001),
    "model__bootstrap": [True, False],
    "model__criterion": ["gini", "entropy"]
}

# Busca aleatória
random_search_rf = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist_rf,
    n_iter=5,
    scoring=f1,
    cv=skf,
    random_state=5762,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

# Treinamento e busca
random_search_rf.fit(X, y_encoded)

# Resultados
print("Random Forest - Melhor F1:", random_search_rf.best_score_)
print("Random Forest - Melhores parâmetros:", random_search_rf.best_params_)
print("Random Forest - Melhor estimador:")
print(random_search_rf.best_estimator_)


Random Forest - Melhor F1: 0.39575200232589164
Random Forest - Melhores parâmetros: {'model__bootstrap': False, 'model__criterion': 'gini', 'model__n_estimators': 797}
Random Forest - Melhor estimador:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['V1', 'V3', 'V4', 'V5', 'V6',
                                                   'V7', 'V8', 'V9', 'V10']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                              

In [7]:

resultados = pd.DataFrame([
    {
        "Modelo": "SVC",
        "Best Score (F1)": random_search_svc.best_score_,
        "Best Params": random_search_svc.best_params_,
        "Best Estimator": str(random_search_svc.best_estimator_)
    },
    {
        "Modelo": "Random Forest",
        "Best Score (F1)": random_search_rf.best_score_,
        "Best Params": random_search_rf.best_params_,
        "Best Estimator": str(random_search_rf.best_estimator_)
    }
])

display(resultados[["Modelo", "Best Score (F1)", "Best Params"]])

# Identificar o melhor resultado médio de validação cruzada
if random_search_svc.best_score_ > random_search_rf.best_score_:
    print("Melhor modelo pela média de F1: SVC")
elif random_search_rf.best_score_ > random_search_svc.best_score_:
    print("Melhor modelo pela média de F1: Random Forest")
else:
    print("Os modelos apresentaram o mesmo F1 médio.")


,Modelo,Best Score (F1),Best Params
0,SVC,0.203083,"{'model__C': 5.539145131649216, 'model__kernel..."
1,Random Forest,0.395752,"{'model__bootstrap': False, 'model__criterion'..."


Melhor modelo pela média de F1: Random Forest
